# Heung-Min Son: Player Intelligence Case Study
### Premier League 2015/2016 — Attacking Analysis from StatsBomb Open Data

A data-driven case study of Heung-Min Son's attacking performance in the 2015/2016 Premier League,
built entirely from verified StatsBomb Open Data events. No statistics are fabricated or estimated
from outside sources.

## Section 1 — Project Overview

### Objective
This notebook performs a focused, data-driven investigation of Heung-Min Son's attacking
contribution for Tottenham Hotspur in the English Premier League 2015/2016 season. It extends
the season-level exploratory analysis in `notebooks/01_football_season_intelligence.ipynb` with a
dedicated player case study.

### Research questions
1. Where does Son generate his shots and attacking actions on the pitch?
2. How does his shot quality (xG) compare with his actual goal output?
3. What kinds of actions (shots, passes, carries, dribbles) contribute to his attacking involvement?
4. How does his per-90 attacking profile compare with other Premier League attackers?
5. What can — and cannot — be concluded from this sample?

### Why these questions matter
A player case study answers the questions a recruitment or performance analyst asks: how productive
is a player per unit of playing time, where is he most dangerous, and does his output match the
quality of chances he creates? Per-90 normalization and xG comparison are standard tools for fair
comparison across players with different minutes.

### Dataset and selection
Source: StatsBomb Open Data — https://github.com/statsbomb/open-data
Selected competition-season: **England — Premier League — 2015/2016**
(competition_id 2, season_id 27), the same season analysed in Project 1.
Heung-Min Son (StatsBomb player ID 3083) played for Tottenham Hotspur in this season and appears
in the lineup and event data with sufficient coverage for a meaningful case study.

### Tools used
Python 3.11, pandas, NumPy, Matplotlib, Seaborn, mplsoccer, requests. Reusable code in `src/`
(data_loader, data_validation, football_metrics, visualizations).

### Scope and limitations
- Only one season (380 matches) is analysed.
- Event data records on-ball actions only; off-ball movement and full tactical context are not
  observed.
- Minutes played are derived from lineup position intervals and treated as approximate.
- xG is StatsBomb's `statsbomb_xg` estimate, not ground truth; no xG model is trained here.
- Sample size is limited to ~28 appearances and ~1,105 minutes for Son.

In [1]:
import warnings
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import requests

from IPython.display import display, HTML

# Resolve the project root from the current working directory or its parents.
_PROJECT_ROOT = None
for _cand in [Path.cwd()] + list(Path.cwd().parents):
    try:
        if _cand.is_dir() and (_cand / "requirements.txt").exists():
            _PROJECT_ROOT = _cand
            break
    except (OSError, RuntimeError):
        continue
if _PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root (requirements.txt)."
    )
PROJECT_ROOT = _PROJECT_ROOT
print("Project root: " + str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from data_loader import load_matches, load_season_events, PITCH_LENGTH, PITCH_WIDTH
from football_metrics import build_match_table, build_shot_table, build_pass_table, build_player_table, minutes_from_lineups
from visualizations import setup_style, save_fig, plot_player_shot_map, plot_player_shot_outcomes, plot_player_heatmap, plot_player_match_by_match, plot_player_comparison

setup_style()
warnings.filterwarnings("always")
np.random.seed(42)

print("Python: " + str(__import__("sys").version.split()[0]))
print("pandas: " + pd.__version__)
print("numpy: " + np.__version__)
print("matplotlib: " + matplotlib.__version__)
print("seaborn: " + sns.__version__)
print("mplsoccer: " + __import__("mplsoccer").__version__)
print("StatsBomb pitch: %.0f x %.0f" % (PITCH_LENGTH, PITCH_WIDTH))

Project root: D:\Projects\Football\Football season intelligence\football-season-intelligence


Python: 3.11.9
pandas: 3.0.6
numpy: 2.4.6
matplotlib: 3.11.2
seaborn: 0.13.2
mplsoccer: 1.8.1
StatsBomb pitch: 120 x 80


In [2]:
CACHE_DIR = PROJECT_ROOT / "data" / "raw"
print("Loading Premier League 2015/2016 match metadata ...")
matches = load_matches(2, 27, CACHE_DIR)
print("Matches loaded: " + str(len(matches)))

print("Loading event files for all matches ...")
events_by_match = load_season_events(
    [m["match_id"] for m in matches], CACHE_DIR, verbose=False
)
print("Matches with events: " + str(len(events_by_match)) + " / " + str(len(matches)))

print("Loading lineup files for all matches ...")
from data_loader import fetch_json, LINEUPS_TEMPLATE, BASE_URL
lineups_by_match = {}
for m in matches:
    mid = int(m["match_id"])
    try:
        lu = fetch_json(
            BASE_URL + "/" + LINEUPS_TEMPLATE.format(match_id=mid),
            CACHE_DIR / "lineups" / (str(mid) + ".json"),
        )
        lineups_by_match[mid] = lu
    except Exception as exc:
        pass
print("Lineups loaded: " + str(len(lineups_by_match)) + " / " + str(len(matches)))

Loading Premier League 2015/2016 match metadata ...
Matches loaded: 380
Loading event files for all matches ...


Matches with events: 380 / 380
Loading lineup files for all matches ...


Lineups loaded: 380 / 380


In [3]:
SON_ID = 3083
SON_NAME = "Heung-Min Son"

# Identify matches where Son appeared (in a Tottenham lineup)
son_appearances = []
for mid, lineups in lineups_by_match.items():
    for team_entry in lineups:
        if team_entry.get("team_name") != "Tottenham Hotspur":
            continue
        for player in team_entry.get("lineup", []) or []:
            if player.get("player_id") == SON_ID:
                positions = player.get("positions", []) or []
                start_reason = positions[0].get("start_reason") if positions else None
                total_sec = 0.0
                for spell in positions:
                    from_ts = spell.get("from")
                    to_ts = spell.get("to")
                    start_sec = 0.0
                    if from_ts:
                        parts = str(from_ts).split(":")
                        start_sec = int(parts[0]) * 60 + int(parts[1])
                    end_sec = 90.0 * 60.0
                    if to_ts:
                        parts = str(to_ts).split(":")
                        end_sec = int(parts[0]) * 60 + int(parts[1])
                    total_sec += max(0.0, end_sec - start_sec)
                mins = round(total_sec / 60.0, 1)
                match_row = next(m for m in matches if m["match_id"] == mid)
                son_appearances.append({
                    "match_id": mid,
                    "date": match_row.get("match_date"),
                    "start_reason": start_reason,
                    "minutes": mins,
                })
                break

son_appearances.sort(key=lambda x: x["match_id"])
played = [a for a in son_appearances if a["minutes"] > 0]
started = [a for a in son_appearances if a["start_reason"] == "Starting XI"]
sub_on = [a for a in son_appearances if a["start_reason"] == "Substitution - On (Tactical)"]
total_minutes = round(sum(a["minutes"] for a in played), 1)

print("=== Heung-Min Son — Player Profile ===")
print("Player ID: " + str(SON_ID))
print("Name: " + SON_NAME)
print("Team: Tottenham Hotspur")
print("Competition: English Premier League 2015/2016")
print("Season source: StatsBomb Open Data (competition_id 2, season_id 27)")
print("Lineup entries: " + str(len(son_appearances)))
print("Appearances played (minutes > 0): " + str(len(played)))
print("Started: " + str(len(started)))
print("Substitute on: " + str(len(sub_on)))
print("Bench / unused sub: " + str(len(son_appearances) - len(played)))
print("Total minutes (derived from lineups): " + str(total_minutes))
print("Matches range: " + son_appearances[0]["date"] + " to " + son_appearances[-1]["date"])

# Build match-level tables
match_table = build_match_table(matches)
shot_table = build_shot_table(events_by_match)
pass_table = build_pass_table(events_by_match)

# Compute minutes played from lineups for per-90 metrics
minutes_by_player = minutes_from_lineups(lineups_by_match)
player_table = build_player_table(events_by_match, minutes_by_player)

# Filter to Son's rows
son_shots = shot_table[shot_table["player_id"] == SON_ID].copy()
son_passes = pass_table[pass_table["player_id"] == SON_ID].copy()
son_player_row = player_table[player_table["player_id"] == SON_ID]

print("\n=== Attacking Performance ===")
print("Total shots: " + str(len(son_shots)))
print("Goals: " + str(int(son_shots["is_goal"].sum())))
print("Shots on target: " + str(int(son_shots["is_on_target"].sum())))
print("Shot conversion: {:.1f}%".format(100 * int(son_shots["is_goal"].sum()) / len(son_shots)))
print("Shot on target rate: {:.1f}%".format(100 * son_shots["is_on_target"].mean()))
print("Total xG: {:.2f}".format(son_shots["xg"].sum()))
print("xG per shot: {:.4f}".format(son_shots["xg"].mean()))
print("Goals minus xG: {:.2f}".format(int(son_shots["is_goal"].sum()) - son_shots["xg"].sum()))
print("Total passes: " + str(len(son_passes)))
print("Passes completed: " + str(int(son_passes["is_complete"].sum())))
print("Pass completion: {:.1f}%".format(100 * son_passes["is_complete"].mean()))
print("Minutes played: " + str(total_minutes))
print("Shots per 90: {:.2f}".format(len(son_shots) / total_minutes * 90))
print("Goals per 90: {:.2f}".format(int(son_shots["is_goal"].sum()) / total_minutes * 90))

=== Heung-Min Son — Player Profile ===
Player ID: 3083
Name: Heung-Min Son
Team: Tottenham Hotspur
Competition: English Premier League 2015/2016
Season source: StatsBomb Open Data (competition_id 2, season_id 27)
Lineup entries: 30
Appearances played (minutes > 0): 28
Started: 13
Substitute on: 15
Bench / unused sub: 2
Total minutes (derived from lineups): 1104.6
Matches range: 2016-05-08 to 2016-03-02



=== Attacking Performance ===
Total shots: 38
Goals: 4
Shots on target: 13
Shot conversion: 10.5%
Shot on target rate: 34.2%
Total xG: 3.85
xG per shot: 0.1014
Goals minus xG: 0.15
Total passes: 455
Passes completed: 359
Pass completion: 78.9%
Minutes played: 1104.6
Shots per 90: 3.10
Goals per 90: 0.33


In [4]:
from visualizations import plot_player_shot_map, plot_player_shot_outcomes

fig_shot_map = plot_player_shot_map(
    son_shots,
    SON_NAME,
    PROJECT_ROOT / "outputs" / "figures" / "son_shot_map.png",
    title="Shot map — Heung-Min Son, Tottenham Hotspur, PL 2015/2016",
)
print("Saved: " + str(fig_shot_map))

fig_shot_outcomes = plot_player_shot_outcomes(
    son_shots,
    SON_NAME,
    PROJECT_ROOT / "outputs" / "figures" / "son_shot_outcomes.png",
    title="Shot outcomes — Heung-Min Son, PL 2015/2016",
)
print("Saved: " + str(fig_shot_outcomes))

# Shot location summary
print("\nShot location summary:")
print("Shots from inside penalty area (x >= 102): " + str(int((son_shots["shot_x"] >= 102).sum())))
print("Goals from inside penalty area: " + str(int(son_shots[(son_shots["shot_x"] >= 102) & (son_shots["is_goal"])].shape[0])))
print("Average shot distance from goal (120 - x): {:.1f} m".format((120 - son_shots["shot_x"]).mean()))

# xG vs goals
print("\nGoals: " + str(int(son_shots["is_goal"].sum())))
print("Total xG: {:.2f}".format(son_shots["xg"].sum()))
print("Goals minus xG: {:.2f}".format(int(son_shots["is_goal"].sum()) - son_shots["xg"].sum()))

Saved: D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\son_shot_map.png


Saved: D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\son_shot_outcomes.png

Shot location summary:
Shots from inside penalty area (x >= 102): 21
Goals from inside penalty area: 4
Average shot distance from goal (120 - x): 16.5 m

Goals: 4
Total xG: 3.85
Goals minus xG: 0.15


In [5]:
from visualizations import plot_player_heatmap

fig_heatmap = plot_player_heatmap(
    events_by_match,
    SON_ID,
    PROJECT_ROOT / "outputs" / "figures" / "son_attacking_heatmap.png",
    title="Event heatmap — Heung-Min Son, PL 2015/2016",
)
print("Saved: " + str(fig_heatmap))

# Event-type breakdown by location zone
from collections import Counter
event_locations = []
for events in events_by_match.values():
    for event in events:
        p = event.get("player")
        if p and p.get("id") == SON_ID:
            loc = event.get("location")
            if isinstance(loc, list) and len(loc) >= 2:
                x = float(loc[0])
                zone = "attacking_third" if x >= 80 else ("middle_third" if x >= 40 else "defensive_third")
                event_locations.append((event.get("type", {}).get("name"), zone))

zone_counts = Counter(event_locations)
print("\nEvent locations by third of pitch:")
for zone in ["defensive_third", "middle_third", "attacking_third"]:
    zone_events = [(t, z) for (t, z) in event_locations if z == zone]
    event_types = Counter(t for t, z in zone_events)
    print("  " + zone + ": " + str(len(zone_events)) + " events, top types: " + str(event_types.most_common(4)))

Saved: D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\son_attacking_heatmap.png



Event locations by third of pitch:
  defensive_third: 224 events, top types: [('Pressure', 70), ('Ball Receipt*', 33), ('Pass', 31), ('Carry', 31)]
  middle_third: 928 events, top types: [('Ball Receipt*', 246), ('Pass', 188), ('Carry', 188), ('Pressure', 146)]
  attacking_third: 1099 events, top types: [('Ball Receipt*', 371), ('Carry', 250), ('Pass', 236), ('Pressure', 82)]


In [6]:
from visualizations import plot_pass_map

# Son pass completion detail
son_pass_complete = int(son_passes["is_complete"].sum())
son_pass_total = len(son_passes)
son_pass_incomplete = son_pass_total - son_pass_complete
print("\n=== Passing ===")
print("Passes attempted: " + str(son_pass_total))
print("Passes completed: " + str(son_pass_complete))
print("Passes incomplete: " + str(son_pass_incomplete))
print("Pass completion: {:.1f}%".format(100 * son_pass_complete / son_pass_total))

# Check for goal-assist and shot-assist tags on Son passes
son_assisted_shots = 0
son_goal_assists = 0
for mid, events in events_by_match.items():
    for e in events:
        p = e.get("player")
        if p and p.get("id") == SON_ID:
            t = e.get("type", {}).get("name")
            if t == "Pass":
                pbody = e.get("pass", {}) or {}
                if pbody.get("goal_assist"):
                    son_goal_assists += 1
                if pbody.get("shot_assist"):
                    son_assisted_shots += 1
print("Goal assists (pass goal_assist tag): " + str(son_goal_assists))
print("Shot assists (pass shot_assist tag): " + str(son_assisted_shots))

# Carries and dribbles
son_carries = 0
son_dribbles = 0
for events in events_by_match.values():
    for e in events:
        p = e.get("player")
        if p and p.get("id") == SON_ID:
            t = e.get("type", {}).get("name")
            if t == "Carry":
                son_carries += 1
            elif t == "Dribble":
                son_dribbles += 1
print("Carry events: " + str(son_carries))
print("Dribble events: " + str(son_dribbles))


=== Passing ===
Passes attempted: 455
Passes completed: 359
Passes incomplete: 96
Pass completion: 78.9%


Goal assists (pass goal_assist tag): 1
Shot assists (pass shot_assist tag): 17


Carry events: 469
Dribble events: 49


In [7]:
from visualizations import plot_player_match_by_match

fig_match = plot_player_match_by_match(
    son_shots,
    son_passes,
    SON_ID,
    SON_NAME,
    PROJECT_ROOT / "outputs" / "figures" / "son_match_by_match_performance.png",
    title="Match-by-match performance — Heung-Min Son, PL 2015/2016",
)
print("Saved: " + str(fig_match))

# Match-level table for Son
match_rows = []
for a in played:
    mid = a["match_id"]
    s = son_shots[son_shots["match_id"] == mid]
    p = son_passes[son_passes["match_id"] == mid]
    goals = int(s["is_goal"].sum())
    shots = int(len(s))
    xg = float(s["xg"].sum())
    passes = int(len(p))
    passes_comp = int(p["is_complete"].sum())
    match_rows.append({
        "match_id": mid,
        "date": a["date"],
        "minutes": a["minutes"],
        "shots": shots,
        "goals": goals,
        "xg": round(xg, 2),
        "passes": passes,
        "pass_comp_pct": round(100 * passes_comp / passes, 1) if passes else None,
    })
son_match_df = pd.DataFrame(match_rows)
print("\n=== Match-by-match table (matches played) ===")
display(son_match_df)

# Strongest and weakest by goals + xG combined
son_match_df["attack_score"] = son_match_df["goals"] + son_match_df["xg"]
best = son_match_df.sort_values("attack_score", ascending=False).head(3)
worst = son_match_df.sort_values("attack_score", ascending=True).head(3)
print("\nStrongest attacking performances (goals + xG):")
for _, r in best.iterrows():
    print("  " + str(r["date"]) + " | " + str(r["minutes"]) + " min | " + str(int(r["goals"])) + " goals | xG {:.2f}".format(r["xg"]) + " | " + str(int(r["shots"])) + " shots")
print("\nWeakest attacking performances (goals + xG):")
for _, r in worst.iterrows():
    print("  " + str(r["date"]) + " | " + str(r["minutes"]) + " min | " + str(int(r["goals"])) + " goals | xG {:.2f}".format(r["xg"]) + " | " + str(int(r["shots"])) + " shots")

Saved: D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\son_match_by_match_performance.png

=== Match-by-match table (matches played) ===


,match_id,date,minutes,shots,goals,xg,passes,pass_comp_pct
0,3753992,2016-05-08,64.5,1,1,0.58,32,84.4
1,3754027,2016-01-16,2.8,0,0,0.00,4,75.0
2,3754057,2015-12-26,11.9,0,0,0.00,4,50.0
3,3754065,2016-01-13,8.9,0,0,0.00,2,50.0
4,3754080,2016-02-06,18.5,1,0,0.06,8,75.0
5,3754087,2016-04-02,65.9,3,0,0.44,34,70.6
6,3754092,2016-05-02,64.1,5,1,0.81,20,85.0
7,3754093,2015-11-29,74.1,4,0,0.31,33,66.7
8,3754109,2015-11-22,84.5,3,0,0.21,30,90.0
9,3754140,2016-01-03,21.3,1,0,0.05,9,88.9



Strongest attacking performances (goals + xG):
  2016-05-02 | 64.1 min | 1 goals | xG 0.81 | 5 shots
  2016-05-08 | 64.5 min | 1 goals | xG 0.58 | 1 shots
  2015-09-20 | 78.8 min | 1 goals | xG 0.25 | 5 shots

Weakest attacking performances (goals + xG):
  2016-01-16 | 2.8 min | 0 goals | xG 0.00 | 0 shots
  2015-12-26 | 11.9 min | 0 goals | xG 0.00 | 0 shots
  2016-01-13 | 8.9 min | 0 goals | xG 0.00 | 0 shots


In [8]:
from visualizations import plot_player_comparison

# Build comparison set: attackers with at least 5 shots and minutes available
pt = player_table.copy()
if "minutes_played" in pt.columns:
    pt_with_min = pt[pt["minutes_played"].notna() & (pt["minutes_played"] > 0)].copy()
    pt_with_min["shots_per_90"] = pt_with_min["shots"] / pt_with_min["minutes_played"] * 90.0
    pt_with_min["goals_per_90"] = pt_with_min["goals"] / pt_with_min["minutes_played"] * 90.0
    pt_with_min["xg_per_90"] = pt_with_min["xg"] / pt_with_min["minutes_played"] * 90.0
else:
    pt_with_min = pt.copy()

attackers = pt_with_min[pt_with_min["shots"] >= 5].copy()
print("\n=== Contextual comparison ===")
print("Attackers with >= 5 shots and minutes data: " + str(len(attackers)))

# Son position in comparison
son_row = attackers[attackers["player_id"] == SON_ID]
if not son_row.empty:
    sr = son_row.iloc[0]
    print("Son shots per 90: {:.2f}".format(sr["shots_per_90"]))
    print("Son goals per 90: {:.2f}".format(sr["goals_per_90"]))
    print("Son xG per 90: {:.2f}".format(sr["xg_per_90"]))
    print("Son goals per 90 rank: " + str(int((attackers["goals_per_90"] > sr["goals_per_90"]).sum()) + 1) + " of " + str(len(attackers)))
    print("Son xG per 90 rank: " + str(int((attackers["xg_per_90"] > sr["xg_per_90"]).sum()) + 1) + " of " + str(len(attackers)))

fig_comp = plot_player_comparison(
    attackers,
    SON_NAME,
    PROJECT_ROOT / "outputs" / "figures" / "son_player_comparison.png",
    title="Goals per 90 — Heung-Min Son vs Premier League attackers (min 5 shots)",
    top_n=12,
)
print("Saved: " + str(fig_comp))

# Top comparison table
comp_show = attackers.sort_values("goals_per_90", ascending=False).head(10)
display_cols = ["player", "team", "shots", "goals", "xg", "minutes_played", "goals_per_90", "xg_per_90"]
print("\nTop attackers by goals per 90:")
display(comp_show[display_cols])


=== Contextual comparison ===
Attackers with >= 5 shots and minutes data: 349
Son shots per 90: 3.10
Son goals per 90: 0.33
Son xG per 90: 0.31
Son goals per 90 rank: 52 of 349
Son xG per 90 rank: 46 of 349


Saved: D:\Projects\Football\Football season intelligence\football-season-intelligence\outputs\figures\son_player_comparison.png

Top attackers by goals per 90:


,player,team,shots,goals,xg,minutes_played,goals_per_90,xg_per_90
31,Kelechi Promise Iheanacho,Manchester City,29,8,5.05,760.6,0.946621,0.597555
2,Sergio Leonel Agüero del Castillo,Manchester City,118,24,17.67,2365.9,0.912972,0.672175
30,Daniel Andre Sturridge,Liverpool,48,8,5.12,981.8,0.733347,0.469342
1,Jamie Vardy,Leicester City,118,24,22.30,3169.5,0.681496,0.633223
68,Divock Okoth Origi,Liverpool,24,5,2.31,668.0,0.673653,0.311228
0,Harry Kane,Tottenham Hotspur,158,25,21.64,3365.8,0.668489,0.578644
94,Daniel Nii Tackie Mensah Welbeck,Arsenal,20,4,3.02,573.2,0.628053,0.474180
5,Olivier Giroud,Arsenal,103,16,13.02,2428.4,0.592983,0.482540
70,Callum Wilson,AFC Bournemouth,18,5,3.28,767.4,0.586396,0.384676
25,Andy Carroll,West Ham United,55,9,6.28,1438.1,0.563243,0.393019


## Section 10 — Analytical Conclusions

### Q1. Where does Son generate his shots and attacking actions?
Son's shots are concentrated in and around the penalty area (see `son_shot_map.png`),
consistent with a forward operating in central attacking zones. His event heatmap
(`son_attacking_heatmap.png`) shows the densest activity in the final third, particularly
in the right and central attacking areas.

### Q2. How does his shot quality compare with his actual goal output?
Son recorded **4 goals from 38 shots** (10.5% conversion) with a total xG of **3.85**.
Goals minus xG = **+0.15**, meaning he scored essentially in line with the expected quality
of his chances — neither significantly over- nor under-performing his xG over this sample.

### Q3. What kinds of actions contribute to his attacking involvement?
Beyond shooting, Son completed hundreds of passes (78.9% completion) and recorded
carry and dribble events (computed from the data, see Section 9). His attacking
involvement is a mix of shooting, passing, and progressive ball-carrying rather than
shooting alone.

### Q4. How does his profile compare with comparable players?
Among Premier League attackers with at least 5 shots and minutes data, Son's per-90
goals and xG rankings are computed in Section 9 (see `son_player_comparison.png`).
His per-90 output is meaningful given ~1,105 minutes across 28 appearances, but the
sample is still modest relative to a full season.

### Q5. What can and cannot be concluded?
Can conclude: Son was an active, central attacking threat for Tottenham in PL 2015/2016,
with shot volume and xG broadly aligned with his goal output, and solid passing volume
for an attacker.
Cannot conclude: finishing skill (sample too small), tactical value (event data lacks
off-ball context), or how this season compares to others (only one season analysed).

### Caveats
- Minutes are derived from lineup position intervals, treated as approximate.
- Per-90 rates divide by derived minutes; treat as indicative, not definitive.
- Goals and assists depend on teammates and game state, not captured here.
- Comparison set is limited to players in this one season with shot and minutes data.